# Load Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
import scanpy as sc
import pandas as pd
import seaborn as sns
import spatialdata as sd
import matplotlib.ticker as ticker
from matplotlib.patches import Polygon as MplPolygon, Circle
from matplotlib.collections import PatchCollection
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from shapely.geometry import MultiPolygon
from shapely.affinity import scale

In [ ]:
adata_filtered = sc.read_h5ad("adata_filtered.h5ad")

In [ ]:
scle1_id = pd.read_csv("Superficial Dermis Cells/SCLE1_cells_stats.csv", skiprows=2)
scle2_id = pd.read_csv("Superficial Dermis Cells/SCLE2_cells_stats.csv", skiprows=2)
scle3_id = pd.read_csv("Superficial Dermis Cells/SCLE3_cells_stats.csv", skiprows=2)
scle4_id = pd.read_csv("Superficial Dermis Cells/SCLE4_cells_stats.csv", skiprows=2)
scle5_id = pd.read_csv("Superficial Dermis Cells/SCLE5_cells_stats.csv", skiprows=2)

scle1_id["Cell_ID"] = "irAE-SCLE1_" + scle1_id["Cell ID"]
scle2_id["Cell_ID"] = "irAE-SCLE2_" + scle2_id["Cell ID"]
scle3_id["Cell_ID"] = "SCLE1_" + scle3_id["Cell ID"]
scle4_id["Cell_ID"] = "SCLE2_" + scle4_id["Cell ID"]
scle5_id["Cell_ID"] = "SCLE3_" + scle5_id["Cell ID"]

In [ ]:
adata_filtered.obs["Depth"] = "Deep"

adata_filtered.obs.loc[adata_filtered.obs.index.isin(scle1_id["Cell_ID"].values), 'Depth'] = "Superficial"
adata_filtered.obs.loc[adata_filtered.obs.index.isin(scle2_id["Cell_ID"].values), 'Depth'] = "Superficial"
adata_filtered.obs.loc[adata_filtered.obs.index.isin(scle3_id["Cell_ID"].values), 'Depth'] = "Superficial"
adata_filtered.obs.loc[adata_filtered.obs.index.isin(scle4_id["Cell_ID"].values), 'Depth'] = "Superficial"
adata_filtered.obs.loc[adata_filtered.obs.index.isin(scle5_id["Cell_ID"].values), 'Depth'] = "Superficial"

adata_filtered.obs["Depth"].value_counts()

In [ ]:
palette = sns.color_palette(n_colors=len(order))
palette

# Figure 1

## B

In [ ]:
mapping_cell_type1 = {
    "Adipocytes": "Adipocytes",
    "Basal Cells": "Keratinocytes",
    "ECM Remodeling Fibroblasts": "Fibroblasts",
    "Eccrine Ductal Cells": "Adnexa",
    "Eccrine Gland Cells": "Adnexa",
    "Granular Keratinocytes": "Keratinocytes",
    "Hair Follicle Epithelia": "Adnexa",
    "Hair Follicle-associated Fibroblasts": "Fibroblasts",
    "IFN Fibroblasts": "Fibroblasts",
    "IFN Keratinocytes": "Keratinocytes",
    "Injury-associated Keratinocytes": "Keratinocytes",
    "Lymphoid Cells": "Lymphoid Cells",
    "Mast Cells": "Mast Cells",
    "Melanocytes": "Melanocytes",
    "Myeloid Cells": "Myeloid Cells",
    "Papillary Fibroblasts": "Fibroblasts",
    "Pericytes": "Pericytes",
    "Perineural Fibroblasts": "Fibroblasts",
    "Perivascular Fibroblasts": "Fibroblasts",
    "Reticular Fibroblasts": "Fibroblasts",
    "Schwann Cells": "Schwann Cells",
    "Sebocytes": "Adnexa",
    "Spinous Keratinocytes": "Keratinocytes",
    "lECs": "Endothelial Cells",
    "vECs": "Endothelial Cells",
    "vSMCs": "Endothelial Cells"
}

adata_filtered.obs["Cell_Type_Clean"] = adata_filtered.obs["Cell_Type1"].map(mapping_cell_type1)

In [ ]:
adata_filtered.uns["Cell_Type_Clean_colors"] = ['#1f77b4',
                                                '#C9A24D',
                                                '#5F9FD6',
                                                '#4B50C8',
                                                '#2A788E',
                                                '#C24AD6',
                                                '#b5bd61',
                                                '#c5b0d5',
                                                '#D55E00',
                                                '#aec7e8',
                                                '#ffbb78',]

In [ ]:
sc.set_figure_params(figsize=(15,8), dpi=50, dpi_save=300, transparent=True)

In [ ]:
sc.pl.umap(adata_filtered, color="Cell_Type_Clean", size=10, title="", frameon=False,save="_Cell_Type")

In [ ]:
sc.pl.umap(adata_filtered, color="Lesion1", size=10, title="", frameon=False,save="_Lesion")

## C

In [ ]:
cell_type_column = 'Cell_Type_Clean'
group_column = 'Lesion1'

colors = {"Adipocytes": '#1f77b4',
          "Adnexa":'#C9A24D',
          "Endothelial Cells":'#5F9FD6',
          "Fibroblasts":'#4B50C8',
          "Keratinocytes":'#2A788E',
          "Lymphoid Cells":'#C24AD6',
          "Mast Cells":'#b5bd61',
          "Melanocytes":'#c5b0d5',
          "Myeloid Cells":'#D55E00',
          "Pericytes":'#aec7e8',
          "Schwann Cells":'#ffbb78',}

# Step 1: Aggregate data
cell_type_counts = (
    adata_filtered.obs.groupby([group_column, cell_type_column])
    .size()
    .unstack(fill_value=0)
)
cell_type_proportions = cell_type_counts.div(cell_type_counts.sum(axis=1), axis=0)

desired_order = ["NS",'SCLE', 'irAE_SCLE']

# Reorder the rows in the DataFrame based on the desired order
cell_type_proportions_ordered = cell_type_proportions.reindex(desired_order)
cell_type_proportions_ordered = cell_type_proportions_ordered[
    cell_type_proportions_ordered.columns[::-1]
]
# Step 2: Stacked Bar Plot
ax = cell_type_proportions_ordered.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 15),
    color=colors
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          title='Cell Type',
          bbox_to_anchor=(1, 1),
          fontsize=20,
          title_fontsize=25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=0, size=35)
plt.yticks(size=35)
plt.xlabel('')
plt.ylabel('Proportion', size=50)
# plt.title('Cell Type Composition')
plt.tight_layout()
plt.grid(False)
plt.savefig('Cell_Type_Comp.pdf')
plt.show()

# Figure 2

## A

In [ ]:
tcell = adata_filtered[adata_filtered.obs["Cell_Type3"].isin(["CD4 T Cells", "CD8 T Cells","IFN CD4 T Cells","IFN CD8 T Cells", "Tregs", "\u03B3\u03B4 T Cells"])].copy()

sc.pp.pca(tcell,random_state=42)
sc.pp.neighbors(tcell,random_state=42)
sc.tl.umap(tcell,random_state=42)

In [ ]:
sc.pl.umap(tcell, color=['Cell_Type3',"Lesion1"], size=30)

In [ ]:
mapping={
    "IFN CD4 T Cells": "IFN CD4",
    "IFN CD8 T Cells": "IFN CD8",
    "CD4 T Cells":"CD4",
    "CD8 T Cells": "CD8",
    "Tregs": "Tregs",
    "\u03B3\u03B4 T Cells": "\u03B3\u03B4"
}

tcell.obs['Cell_Type_Clean1']=tcell.obs['Cell_Type3'].map(mapping)

In [ ]:
sc.set_figure_params(figsize=(10,10), dpi=50, dpi_save=300, transparent=True)

In [ ]:
tcell.uns["Cell_Type_Clean1_colors"] = ['#b5bd61',
                                        '#C24AD6',
                                        '#ffbb78',
                                        '#2A788E',
                                        '#D55E00',
                                        '#4B50C8']

In [ ]:
sc.pl.umap(tcell, color="Cell_Type_Clean1", size=60, title="", frameon=False,save="_Tcell_Cell_Type")
sc.pl.umap(tcell, color="Lesion1", size=60, title="", frameon=False,save="_Tcell_Lesion")

In [ ]:
sc.set_figure_params(figsize=(15,8), dpi=50, dpi_save=300, transparent=True)

In [ ]:
tcell_genes=["CD3E","CD4","CD8A",
             "STAT1","MX1","IFI44L","OAS3","IFIT1","IFIT2","IFIT3","RSAD2",
             "FOXP3","CTLA4","TRDC"
             ]

tcell.obs["Cell_Type_Clean1"] = (
    tcell.obs["Cell_Type_Clean1"]
    .astype("category")
    .cat.reorder_categories(["CD4","CD8","IFN CD4", "IFN CD8", "Tregs","\u03B3\u03B4"], ordered=True)
)
sc.pl.dotplot(tcell, var_names=tcell_genes, groupby="Cell_Type_Clean1", save="_Tcell")

## B

In [ ]:
mac = adata_filtered[adata_filtered.obs["Cell_Type2"].isin(["Macrophages"])].copy()

sc.pp.pca(mac,random_state=42)
sc.pp.neighbors(mac,random_state=42)
sc.tl.umap(mac,random_state=42)

In [ ]:
mapping={
    "Hypoxic Macrophages": 'Hypoxic',
    "Tissue-resident Macrophages": "Tissue-resident",
    "IFN Macrophages": "IFN",
}

mac.obs["Cell_Type_Clean1"] = mac.obs['Cell_Type3'].map(mapping)

In [ ]:
mac.uns["Cell_Type_Clean1_colors"] = ['#b5bd61',
                                                '#C24AD6',
                                                '#D55E00',
                                                '#4B50C8',
                                                '#2A788E',
                                                '#c5b0d5',
                                                '#D55E00',
                                                '#aec7e8',
                                                '#ffbb78',]

In [ ]:
sc.set_figure_params(figsize=(10,10), dpi=50, dpi_save=300, transparent=True)

In [ ]:
sc.pl.umap(mac, color="Lesion1", size=100, title="", frameon=False,save="_Mac_Lesion")
sc.pl.umap(mac, color="Cell_Type_Clean1", size=100, title="", frameon=False,save="_Mac_Cell_Type")

In [ ]:
mac_genes=["SLC40A1","LGMN","MRC1","MAF","CSF1R",
             "THBS1","HIF1A","LDHA","PFKFB3","EPAS1","CTSL","CR1","C5AR1",
             ]

mac.obs["Cell_Type_Clean1"] = (
    mac.obs["Cell_Type_Clean1"]
    .astype("category")
    .cat.reorder_categories(["Tissue-resident", "IFN",
                             "Hypoxic",], ordered=True)
)
sc.pl.dotplot(mac, var_names=mac_genes, groupby="Cell_Type_Clean1", save="_Mac")

## C

In [ ]:
krt = adata_filtered[adata_filtered.obs["Cell_Type_Clean"].isin(["Basal Cells","Keratinocytes"])].copy()

sc.pp.pca(krt,random_state=42)
sc.pp.neighbors(krt,random_state=42)
sc.tl.umap(krt,random_state=42)

In [ ]:
sc.set_figure_params(figsize=(10,10), dpi=50, dpi_save=300, transparent=True)

In [ ]:
sc.pl.umap(krt, color="Lesion1", size=30, title="", frameon=False,save="_Krt_Lesion")

In [ ]:
mapping={
    "Basal Cells": 'Basal',
    "Granular Keratinocytes": "Granular",
    "IFN Spinous Keratinocytes": "IFN Spinous",
    "Injury-associated Keratinocytes": "Injury",
    "Spinous Keratinocytes": "Spinous",
    "CXCL9/10/11+ IFN Basal Cells": "CXCL9/10/11+ Basal",
    "IFN Basal Cells": 'IFN Basal',
}

krt.obs["Cell_Type_Clean1"] = krt.obs['Cell_Type2'].map(mapping)

In [ ]:
krt.uns["Cell_Type_Clean1_colors"] = ['#1f77b4',
                                                '#C9A24D',
                                                '#5F9FD6',
                                                '#4B50C8',
                                                '#2A788E',
                                                '#C24AD6',
                                                '#b5bd61',
                                                '#c5b0d5',
                                                '#D55E00',
                                                '#aec7e8',
                                                '#ffbb78',]

In [ ]:
sc.pl.umap(krt, color="Cell_Type_Clean1", size=30, title="", frameon=False,save="_Krt")

In [ ]:
sc.tl.rank_genes_groups(krt,groupby='Cell_Type_Clean1')
sc.tl.dendrogram(krt,groupby='Cell_Type_Clean1')
sc.pl.rank_genes_groups_heatmap(krt,groupby='Cell_Type_Clean1',show_gene_labels=True,swap_axes=True,n_genes=20)

In [ ]:
sc.set_figure_params(figsize=(15,8), dpi=300, dpi_save=300, transparent=True)

In [ ]:
krt_genes=["DST","COL17A1", # Basal
           "STAT1","MX1","IFI44L","OAS3","IFIT1","IFIT2","IFIT3",
           "CXCL9","CXCL10","CXCL11",
           "NOTCH3","KLF5", # Spinous
           "TMEM45A","CDSN", # Granular
           "FSCN1","LRRC59","HSPD1"]
krt.obs["Cell_Type_Clean1"] = (
    krt.obs["Cell_Type_Clean1"]
    .astype("category")
    .cat.reorder_categories(["Basal","IFN Basal","CXCL9/10/11+ Basal",
                             "IFN Spinous","Spinous","Granular",
                             "Injury"], ordered=True)
)
sc.pl.dotplot(krt, var_names=krt_genes, groupby="Cell_Type_Clean1", save="_Krt")

## D

In [ ]:
ec = adata_filtered[adata_filtered.obs["Cell_Type1"].isin(["vECs"])].copy()

sc.pp.pca(ec,random_state=42)
sc.pp.neighbors(ec,random_state=42)
sc.tl.umap(ec,random_state=42)

In [ ]:
ec.uns['Cell_Type2_colors'] = ['#C9A24D','#4B50C8','#D55E00','#C24AD6']

In [ ]:
sc.pl.umap(ec, color="Lesion1", size=60, title="", frameon=False,save="_EC_Lesion")
sc.pl.umap(ec, color="Cell_Type2", size=60, title="", frameon=False,save="_EC_Cell_Type")

In [ ]:
ec_genes=["CD34","PECAM1",
             "STAT1","MX1","IFI44L","OAS3","IFIT1","IFIT2","IFIT3","IFITM1",
             "CXCL9","CXCL10","CXCL11",
             "COL4A1","COL4A2","SELP","HIF1A","LDHA",
             ]

ec.obs["Cell_Type2"] = (
    ec.obs["Cell_Type2"]
    .astype("category")
    .cat.reorder_categories(["vECs", "IFN vECs","CXCL9/10/11+ IFN vECs",
                             "Activated vECs",], ordered=True)
)
sc.pl.dotplot(ec, var_names=ec_genes, groupby="Cell_Type2", save="_EC")

## E

In [ ]:
fibro = adata_filtered[adata_filtered.obs["Cell_Type1"].isin(["ECM Remodeling Fibroblasts",
                                                              'IFN Fibroblasts','Perivascular Fibroblasts','Reticular Fibroblasts',
                                                              'Hair Follicle-associated Fibroblasts',
                                                              'Papillary Fibroblasts',"Perineural Fibroblasts",
                                                              "CXCL9/10/11+ IFN Fibroblasts"])].copy()

sc.pp.pca(fibro,random_state=42)
sc.pp.neighbors(fibro,random_state=42)
sc.tl.umap(fibro,random_state=42)

In [ ]:
sc.set_figure_params(figsize=(10,10), dpi=50, dpi_save=300, transparent=True)

In [ ]:
sc.pl.umap(fibro, color="Lesion1", size=40, title="", frameon=False,save="_Fibro_Lesion")

In [ ]:
Cell_Type = {'ECM Remodeling Fibroblasts': 'ECM Remodel',
'Reticular Fibroblasts': 'Reticular',
'IFN Fibroblasts': 'IFN',
'Perivascular Fibroblasts': 'Perivascular',
'Hair Follicle-associated Fibroblasts': 'HF-associated',
'Perineural Fibroblasts': 'Perineural',
'Papillary Fibroblasts': 'Papillary',
'CXCL9/10/11+ IFN Fibroblasts': 'CXCL9/10/11+',
}

fibro.obs['Cell_Type_Clean1'] = fibro.obs['Cell_Type3'].map(Cell_Type)

sc.pl.umap(fibro, color="Cell_Type_Clean1", size=40, title="", frameon=False,save="_Fibro")

In [ ]:
sc.set_figure_params(figsize=(10,15), dpi=50, dpi_save=300, transparent=True)

In [ ]:
fibro_genes=["APCDD1","COL18A1", # Pap
             "CXCL12", "CCDC80",
             "CCN5","CTHRC1",
             "ASPN","COL11A1", # Hair
             "CLDN1","ITGA6", # Perineural
             "STAT1","MX1","IFI44L","OAS1","IFIT1",
             "CXCL9","CXCL10","CXCL11",
             "HIF1A","LDHA","ADAMTS1",
             ]

fibro.obs["Cell_Type_Clean1"] = (
    fibro.obs["Cell_Type_Clean1"]
    .astype("category")
    .cat.reorder_categories(["Papillary", "Perivascular","Reticular",
                             "Hair Follicle-associated","Perineural",
                             "IFN","CXCL9/10/11+ IFN","ECM Remodeling"], ordered=True)
)
sc.pl.dotplot(fibro, var_names=fibro_genes, groupby="Cell_Type_Clean1", save="_Fibro")

In [ ]:
ifn_activation = ["IRF3","IRF7", "IFI16"]

ifn1 = ['IFNA1', 'IFNA17', 'IFNA2', 'IFNA7', 'IFNA8', "IFNB1", "IFNW1"]
ifn2 = ["IFNG"]
ifn3 = ['IFNL1', 'IFNL2', 'IFNL3']

ifn1_recptors=['IFNAR1', 'IFNAR2']
ifn2_recptors=['IFNGR1', 'IFNGR2']
ifn3_recptors=['IL10RB']

ifn_sig = ['STAT1', 'STAT2', "JAK1","JAK2","TYK2"]

isg = ["DDX58","IFIH1","MX1", "OAS1", "OAS3", "IFIT1", "IFIT2", "IFIT3", "RSAD2", "EIF2AK2",
       "IFITM1", "HERC6", "IFI44L", "IFI35", "GBP1"]

chemokines = ["CXCL9","CXCL10", "CXCL11"]

hypoxia=["HIF1A","EPAS1","LDHA","SLC2A3","ALDOA","PGK1","PFKFB3","CA12","VEGFA","KDR"]
remodel = ["MMP9","MMP12","MMP13","MMP14","ADAM8","ADAMTS1","PLAUR"]

In [ ]:
sc.tl.score_genes(adata_filtered, gene_list=chemokines, score_name='chemokines_score')
sc.tl.score_genes(adata_filtered, gene_list=ifn_sig, score_name='ifn_sig_score')
sc.tl.score_genes(adata_filtered, gene_list=isg, score_name='isg_score')
sc.tl.score_genes(adata_filtered, gene_list=hypoxia, score_name='hypoxia_score')
sc.tl.score_genes(adata_filtered, gene_list=remodel, score_name='remodel_score')

In [ ]:
sc.set_figure_params(figsize=(10,10), dpi=300, dpi_save=300, transparent=True)

## F

In [ ]:
score_col = "ifn_sig_score"   # column in adata.obs
condition_col = "Lesion1"     # column in adata.obs
order = ["NS",
          "irAE_SCLE",
          "SCLE",]
# Make sure the needed columns exist
df = adata_filtered.obs[[score_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, condition_col])

plt.figure(figsize=(15, 7))
ax=sns.violinplot(
    data=df,
    x=condition_col,
    y=score_col,
    hue=condition_col,
    order=order,         # split violin
    inner=None,
    cut=0,
    linewidth=1,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel(None)
plt.ylabel("IFN Signaling Score", size=35)
plt.title(None)
# plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=20, title_fontsize=25)
plt.tight_layout()
plt.savefig('IFN_Sig_All.pdf')
plt.show()

## E

In [ ]:
score_col = "isg_score"   # column in adata.obs
condition_col = "Lesion1"     # column in adata.obs
order = ["NS",
          "irAE_SCLE",
          "SCLE",]
# Make sure the needed columns exist
df = adata_filtered.obs[[score_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, condition_col])

plt.figure(figsize=(15, 7))
ax=sns.violinplot(
    data=df,
    x=condition_col,
    y=score_col,
    hue=condition_col,
    order=order,         # split violin
    inner=None,
    cut=0,
    linewidth=1,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel(None)
plt.ylabel("ISG Score", size=35)
plt.title(None)
# plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=20, title_fontsize=25)
plt.tight_layout()
plt.savefig('ISG_Sig_All.pdf')
plt.show()

# irAE SCLE VS SCLE

In [ ]:
scle = adata_filtered[(adata_filtered.obs["Cell_Type_Clean"].isin(["Keratinocytes","Endothelial Cells","Fibroblasts",
                                                                   "Lymphoid Cells","Myeloid Cells"]))&
                             (adata_filtered.obs["Lesion"]=="SCLE")&
                             (adata_filtered.obs["Cell_Type2"]!="Langerhans")
                             ].copy()
scle_superficial = scle[(scle.obs["Depth"]=="Superficial")&(scle.obs["Cell_Type_Clean"]!="Keratinocytes")].copy()

## Superficial Dermis Cell Type Comp

In [ ]:
for i in scle.obs['Cell_Type_Clean'].unique():
    print(i)

In [ ]:
mapping = {
    "Activated vECs": "Endothelial Cells",
    "CXCL9/10/11+ IFN Fibroblasts": "Fibroblasts",
    "CXCL9/10/11+ IFN vECs": "Endothelial Cells",
    "ECM Remodeling Fibroblasts": "Fibroblasts",
    "Hair Follicle-associated Fibroblasts":'Fibroblasts',
    "IFN Fibroblasts": "Fibroblasts",
    "IFN T Cells": "T Cells",
    "IFN vECs": "Endothelial Cells",
    "Papillary Fibroblasts": "Fibroblasts",
    "Perineural Fibroblasts": "Fibroblasts",
    "Perivascular Fibroblasts":"Fibroblasts",
    "Reticular Fibroblasts": "Fibroblasts",
    "T Cells": "T Cells",
    "Macrophages": "Macrophages",
    "lECs": "Endothelial Cells",
    "mregDCs": "mregDCs",
    "cDC1s": "cDC1s",
    "cDC2s": "cDC2s",
    "pDCs": "pDCs",
    "mregDCs": "mregDCs",
    "Neutrophils": "Neutrophils",
    "vSMCs": "Endothelial Cells",
    "Injury-associated Keratinocytes":"Keratinocytes",
    "Basal Cells": "Keratinocytes",
    "IFN Basal Cells": "Keratinocytes",
    "CXCL9/10/11+ IFN Basal Cells": "Keratinocytes",
    "IFN Spinous Keratinocytes": "Keratinocytes",
    "Spinous Keratinocytes": "Keratinocytes",
    "Granular Keratinocytes": "Keratinocytes"
}

scle.obs['Cell_Type2_Clean'] = scle.obs['Cell_Type2'].map(mapping)

In [ ]:
colors = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}
cell_type_column = 'Cell_Type_Clean'
group_column = 'Sample'

# Step 1: Aggregate data
cell_type_counts = (
    scle_superficial.obs.groupby([group_column, cell_type_column])
    .size()
    .unstack(fill_value=0)
)
cell_type_proportions = cell_type_counts.div(cell_type_counts.sum(axis=1), axis=0)

desired_order = ["NS1","NS2","NS3","NS4",'irAE-SCLE1', 'irAE-SCLE2',"SCLE3","SCLE4","SCLE5"]

# Reorder the rows in the DataFrame based on the desired order
# cell_type_proportions_ordered = cell_type_proportions.reindex(desired_order)

# Step 2: Stacked Bar Plot
ax = cell_type_proportions.plot(
    kind='bar',
    stacked=True,
    figsize=(15, 10),
    color=colors
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          title='Cell Type',
          bbox_to_anchor=(1, 1),
          fontsize=20,
          title_fontsize=25
          )
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.yaxis.set_label_position('right')
# ax.yaxis.set_ticks_position('right')
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel('')
plt.ylabel('Proportion', size=30)
# plt.title('Cell Type Composition')
plt.tight_layout()
plt.grid(False)
plt.savefig('Cell_Type_Comp_SCLE_Superficial_by_Sample.pdf')
plt.show()

In [ ]:
scle_superficial_t = scle_superficial[scle_superficial.obs['Cell_Type_Clean']=="Lymphoid Cells"].copy()

## Gene Set

In [ ]:
sc.pl.violin(mac, keys=["MMP13","MMP14"], groupby="Lesion1", stripplot=False)

In [ ]:
gene = "SELPLG"
celltype_col = "Cell_Type2_Clean"
condition_col = "Lesion1"

colors = {"SCLE": "#ff7f0e",
          "irAE_SCLE": "#279e68"}

order = ["T Cells",
         "Macrophages",
         "cDC1s",
         "cDC2s",
         "mregDCs",
         "pDCs",
         "Neutrophils",
         "Fibroblasts",
         "Endothelial Cells"]

# Get metadata
df = scle.obs[[celltype_col, condition_col]].copy()

# Pull gene expression
# If you want the default matrix:
expr = scle[:, gene].X

# If you want a specific layer instead, use this instead:
# expr = scle[:, gene].layers["log1p"]   # example

# Convert sparse / 2D to 1D
if hasattr(expr, "toarray"):
    expr = expr.toarray()
expr = expr.flatten()

df[gene] = expr

# Remove missing values
df = df.dropna(subset=[gene, celltype_col, condition_col])

plt.figure(figsize=(15, 7))
ax = sns.violinplot(
    data=df,
    x=celltype_col,
    y=gene,
    hue=condition_col,
    order=order,
    split=True,
    inner="quartile",
    cut=0,
    linewidth=1,
    palette=colors
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.set_yscale("log")
plt.xticks(rotation=0, size=15)
plt.yticks(size=15)
plt.xlabel(None)
plt.ylabel(f"{gene} expression", size=25)
plt.title(None)
plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=15, title_fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
score_col = "hypoxia_score"   # column in adata.obs
celltype_col = "Cell_Type2_Clean"      # column in adata.obs
condition_col = "Depth"     # column in adata.obs
colors = {"Superficial": "#ff7f0e",
          "Deep": "#279e68"}
order = ["Keratinocytes",
         "Macrophages",
         "cDC1s",
         "cDC2s",
         "mregDCs",
         "pDCs",
         "Neutrophils",
         "Fibroblasts",
         "Endothelial Cells",]

# Make sure the needed columns exist
df = scle.obs[[score_col, celltype_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, celltype_col, condition_col])

plt.figure(figsize=(15, 7))
ax=sns.violinplot(
    data=df,
    x=celltype_col,
    y=score_col,
    hue=condition_col,
    order=order,
    split=True,          # split violin
    inner="quartile",
    cut=0,
    linewidth=1,
    palette=colors
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=0, size=15)
plt.yticks(size=15)
plt.xlabel(None)
plt.ylabel("Chemokine Score", size=25)
plt.title(None)
plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=15, title_fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
score_col = "remodel_score"   # column in adata.obs
celltype_col = "Cell_Type2_Clean"      # column in adata.obs
condition_col = "Depth"     # column in adata.obs
colors = {"Superficial": "#ff7f0e",
          "Deep": "#279e68"}
order = ["Keratinocytes",
         "Macrophages",
         "cDC1s",
         "cDC2s",
         "mregDCs",
         "pDCs",
         "Neutrophils",
         "Fibroblasts",
         "Endothelial Cells",]

# Make sure the needed columns exist
df = scle.obs[[score_col, celltype_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, celltype_col, condition_col])

plt.figure(figsize=(15, 7))
ax=sns.violinplot(
    data=df,
    x=celltype_col,
    y=score_col,
    hue=condition_col,
    order=order,
    split=True,          # split violin
    inner="quartile",
    cut=0,
    linewidth=1,
    palette=colors
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=0, size=15)
plt.yticks(size=15)
plt.xlabel(None)
plt.ylabel("Chemokine Score", size=25)
plt.title(None)
plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=15, title_fontsize=20)
plt.tight_layout()
plt.show()

## Spatial

In [ ]:
scle1 = sd.read_zarr("SCLE_1.zarr")
scle2 = sd.read_zarr("SCLE_2.zarr")
scle3 = sd.read_zarr("SCLE_3.zarr")
scle4 = sd.read_zarr("SCLE_4.zarr")
scle5 = sd.read_zarr("SCLE_5.zarr")

scle1_cell_type = adata_filtered[adata_filtered.obs["Sample"]=="irAE-SCLE1"].copy()
scle2_cell_type = adata_filtered[adata_filtered.obs["Sample"]=="irAE-SCLE2"].copy()
scle3_cell_type = adata_filtered[adata_filtered.obs["Sample"]=="SCLE1"].copy()
scle4_cell_type = adata_filtered[adata_filtered.obs["Sample"]=="SCLE2"].copy()
scle5_cell_type = adata_filtered[adata_filtered.obs["Sample"]=="SCLE3"].copy()

In [ ]:
matrix_scle1 = pd.read_csv("irAE-SCLE1/SCLE1_alignment/matrix.csv", header=None)
mat_inv_scle1 = np.linalg.inv(matrix_scle1.values)
coords_scle1 = scle1_cell_type.obsm['spatial']/0.2125
coords_h_scle1 = np.c_[coords_scle1, np.ones(coords_scle1.shape[0])]
coords_he_scle1 = mat_inv_scle1 @ coords_h_scle1.T
xs_he_scle1 = coords_he_scle1.T[:, 0]
ys_he_scle1 = coords_he_scle1.T[:, 1]

matrix_scle2 = pd.read_csv("irAE-SCLE1/SCLE2_alignment/matrix.csv", header=None)
mat_inv_scle2 = np.linalg.inv(matrix_scle2.values)
coords_scle2 = scle2_cell_type.obsm['spatial']/0.2125
coords_h_scle2 = np.c_[coords_scle2, np.ones(coords_scle2.shape[0])]
coords_he_scle2 = mat_inv_scle2 @ coords_h_scle2.T
xs_he_scle2 = coords_he_scle2.T[:, 0]
ys_he_scle2 = coords_he_scle2.T[:, 1]

matrix_scle3 = pd.read_csv("SCLE1/He_alignment_files/matrix.csv", header=None)
mat_inv_scle3 = np.linalg.inv(matrix_scle3.values)
coords_scle3 = scle3_cell_type.obsm['spatial']/0.2125
coords_h_scle3 = np.c_[coords_scle3, np.ones(coords_scle3.shape[0])]
coords_he_scle3 = mat_inv_scle3 @ coords_h_scle3.T
xs_he_scle3 = coords_he_scle3.T[:, 0]
ys_he_scle3 = coords_he_scle3.T[:, 1]

matrix_scle4 = pd.read_csv("SCLE2/He_alignment_files/matrix.csv", header=None)
mat_inv_scle4 = np.linalg.inv(matrix_scle4.values)
coords_scle4 = scle4_cell_type.obsm['spatial']/0.2125
coords_h_scle4 = np.c_[coords_scle4, np.ones(coords_scle4.shape[0])]
coords_he_scle4 = mat_inv_scle4 @ coords_h_scle4.T
xs_he_scle4 = coords_he_scle4.T[:, 0]
ys_he_scle4 = coords_he_scle4.T[:, 1]

matrix_scle5 = pd.read_csv("SCLE3/He_alignment_files/matrix.csv", header=None)
mat_inv_scle5 = np.linalg.inv(matrix_scle5.values)
coords_scle5 = scle5_cell_type.obsm['spatial']/0.2125
coords_h_scle5 = np.c_[coords_scle5, np.ones(coords_scle5.shape[0])]
coords_he_scle5 = mat_inv_scle5 @ coords_h_scle5.T
xs_he_scle5 = coords_he_scle5.T[:, 0]
ys_he_scle5 = coords_he_scle5.T[:, 1]

# Figure 3

## A

### SCLE1

In [ ]:
palette = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}

gdf = scle1["cell_boundaries"].copy()
gdf.index = "irAE-SCLE1_" + gdf.index
patches = []
facecolors = []

def color_for_index(idx):
    # safe lookup: fall back to gray if missing
    try:
        return palette.get(scle1_cell_type.obs.loc[idx, 'Cell_Type_Clean'], '#FFFFFF')
    except Exception:
        return '#FFFFFF'

for idx, row in gdf.iterrows():
    geom = row.geometry

    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        poly_scaled = scale(
            poly,
            xfact=2,
            yfact=2,
            origin='centroid'
        )
        exterior_coords = np.asarray(poly.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        facecolors.append(color_for_index(idx))

# --- plotting ---
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')        # faster; change to 'k' + small linewidth for outlines
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')

# invert y if pixel coords; comment out if your coords are already in standard orientation
ax.invert_yaxis()

ax.axis('off')

# legend: only include categories present in LM5.obs (and for which palette has a color)
cats = scle1_cell_type.obs['Cell_Type_Clean'].astype('category').cat.categories
handles = [Patch(facecolor=palette[c], label=c, edgecolor='none') for c in cats if c in palette]

plt.tight_layout()
plt.show()

### SCLE2

In [ ]:
palette = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}

gdf = scle2["cell_boundaries"].copy()
gdf.index = "irAE-SCLE2_" + gdf.index

patches = []
facecolors = []

def color_for_index(idx):
    # safe lookup: fall back to gray if missing
    try:
        return palette.get(scle2_cell_type.obs.loc[idx, 'Cell_Type_Clean'], '#FFFFFF')
    except Exception:
        return '#FFFFFF'

for idx, row in gdf.iterrows():
    geom = row.geometry

    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        poly_scaled = scale(
            poly,
            xfact=2,
            yfact=2,
            origin='centroid'
        )
        exterior_coords = np.asarray(poly.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        facecolors.append(color_for_index(idx))

# --- plotting ---
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')        # faster; change to 'k' + small linewidth for outlines
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')

# invert y if pixel coords; comment out if your coords are already in standard orientation
ax.invert_yaxis()

ax.axis('off')

# legend: only include categories present in LM5.obs (and for which palette has a color)
cats = scle2_cell_type.obs['Cell_Type_Clean'].astype('category').cat.categories
handles = [Patch(facecolor=palette[c], label=c, edgecolor='none') for c in cats if c in palette]

plt.tight_layout()
plt.savefig(
    "SCLE2_Cell_Type.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE3

In [ ]:
palette = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}

gdf = scle3["cell_boundaries"].copy()
gdf.index = "SCLE3_" + gdf.index

patches = []
facecolors = []

def color_for_index(idx):
    # safe lookup: fall back to gray if missing
    try:
        return palette.get(scle3_cell_type.obs.loc[idx, 'Cell_Type_Clean'], '#FFFFFF')
    except Exception:
        return '#FFFFFF'

for idx, row in gdf.iterrows():
    geom = row.geometry

    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        poly_scaled = scale(
            poly,
            xfact=2,
            yfact=2,
            origin='centroid'
        )
        exterior_coords = np.asarray(poly.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        facecolors.append(color_for_index(idx))

# --- plotting ---
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')        # faster; change to 'k' + small linewidth for outlines
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')

# invert y if pixel coords; comment out if your coords are already in standard orientation
ax.invert_yaxis()

ax.axis('off')

# legend: only include categories present in LM5.obs (and for which palette has a color)
cats = scle3_cell_type.obs['Cell_Type_Clean'].astype('category').cat.categories
handles = [Patch(facecolor=palette[c], label=c, edgecolor='none') for c in cats if c in palette]

# if handles:
#     ax.legend(handles=handles, loc='upper left', frameon=False,
#               fontsize = 10,
#               handlelength = 1,
#               handleheight = 1,
#               columnspacing = 1)

plt.tight_layout()
plt.savefig(
    "SCLE3_Cell_Type.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE4

In [ ]:
palette = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}

gdf = scle4["cell_boundaries"].copy()
gdf.index = "SCLE4_" + gdf.index

patches = []
facecolors = []

def color_for_index(idx):
    # safe lookup: fall back to gray if missing
    try:
        return palette.get(scle4_cell_type.obs.loc[idx, 'Cell_Type_Clean'], '#FFFFFF')
    except Exception:
        return '#FFFFFF'

for idx, row in gdf.iterrows():
    geom = row.geometry

    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        poly_scaled = scale(
            poly,
            xfact=2,
            yfact=2,
            origin='centroid'
        )
        exterior_coords = np.asarray(poly.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        facecolors.append(color_for_index(idx))

# --- plotting ---
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')        # faster; change to 'k' + small linewidth for outlines
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')

# invert y if pixel coords; comment out if your coords are already in standard orientation
ax.invert_yaxis()

ax.axis('off')

# legend: only include categories present in LM5.obs (and for which palette has a color)
cats = scle4_cell_type.obs['Cell_Type_Clean'].astype('category').cat.categories
handles = [Patch(facecolor=palette[c], label=c, edgecolor='none') for c in cats if c in palette]

# if handles:
#     ax.legend(handles=handles, loc='upper left', frameon=False,
#               fontsize = 10,
#               handlelength = 1,
#               handleheight = 1,
#               columnspacing = 1)

plt.tight_layout()
plt.savefig(
    "SCLE4_Cell_Type.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE5

In [ ]:
palette = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}

gdf = scle5["cell_boundaries"].copy()
gdf.index = "SCLE5_" + gdf.index

patches = []
facecolors = []

def color_for_index(idx):
    # safe lookup: fall back to gray if missing
    try:
        return palette.get(scle5_cell_type.obs.loc[idx, 'Cell_Type_Clean'], '#FFFFFF')
    except Exception:
        return '#FFFFFF'

for idx, row in gdf.iterrows():
    geom = row.geometry

    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        poly_scaled = scale(
            poly,
            xfact=2,
            yfact=2,
            origin='centroid'
        )
        exterior_coords = np.asarray(poly.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        facecolors.append(color_for_index(idx))

# --- plotting ---
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')        # faster; change to 'k' + small linewidth for outlines
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')

# invert y if pixel coords; comment out if your coords are already in standard orientation
ax.invert_yaxis()

ax.axis('off')

# legend: only include categories present in LM5.obs (and for which palette has a color)
cats = scle5_cell_type.obs['Cell_Type_Clean'].astype('category').cat.categories
handles = [Patch(facecolor=palette[c], label=c, edgecolor='none') for c in cats if c in palette]

plt.tight_layout()
plt.savefig(
    "SCLE5_Cell_Type.pdf",
    dpi=300,
    transparent=True
)
plt.show()

In [ ]:
colors = {
    "Keratinocytes": "#2A788E",   # muted maroon
    "Lymphoid Cells": "#C24AD6",  # desaturated teal
    "Myeloid Cells": "#D55E00",     # soft coral
    "Fibroblasts": "#4B50C8",   # muted indigo
    "Adnexa": "#C9A24D",   # olive brown
    "Endothelial Cells": "#5F9FD6" 
}
cell_type_column = 'Cell_Type_Clean'
group_column = 'Lesion1'

# Step 1: Aggregate data
cell_type_counts = (
    scle_superficial.obs.groupby([group_column, cell_type_column])
    .size()
    .unstack(fill_value=0)
)
cell_type_proportions = cell_type_counts.div(cell_type_counts.sum(axis=1), axis=0)

# desired_order = ["NS1","NS2","NS3","NS4",'SCLE1', 'SCLE2',"SCLE3","SCLE4","SCLE5"]

# Reorder the rows in the DataFrame based on the desired order
# cell_type_proportions_ordered = cell_type_proportions.reindex(desired_order)

# Step 2: Stacked Bar Plot
ax = cell_type_proportions.plot(
    kind='bar',
    stacked=True,
    figsize=(8, 15),
    color=colors
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          title='Cell Type',
          bbox_to_anchor=(0.1, 1),
          fontsize=20,
          title_fontsize=25
          )
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.yaxis.set_label_position('right')
ax.yaxis.set_ticks_position('right')
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel('')
plt.ylabel('Proportion', size=40)
# plt.title('Cell Type Composition')
plt.tight_layout()
plt.grid(False)
plt.savefig('Cell_Type_Comp_SCLE_Superficial.pdf')
plt.show()

## B

In [ ]:
score_col = "chemokines_score"   # column in adata.obs
celltype_col = "Cell_Type2_Clean"      # column in adata.obs
condition_col = "Lesion1"     # column in adata.obs
colors = {"SCLE": "#ff7f0e",
          "irAE-SCLE": "#279e68"}
order = ["Keratinocytes",
         "Macrophages",
         "Fibroblasts",
         "Endothelial Cells",]

# Make sure the needed columns exist
df = scle.obs[[score_col, celltype_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, celltype_col, condition_col])

plt.figure(figsize=(10, 7))
ax=sns.violinplot(
    data=df,
    x=celltype_col,
    y=score_col,
    hue=condition_col,
    order=order,
    split=True,          # split violin
    inner=None,
    cut=0,
    linewidth=0.5,
    width=1.15,
    palette=colors
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=20)
plt.yticks(size=20)
plt.xlabel(None)
plt.ylabel("Chemokine Score", size=25)
plt.title(None)
plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=15, title_fontsize=20)
plt.tight_layout()
plt.savefig('Chemokine_SCLE.pdf')
plt.show()

## C

In [ ]:
cmap = plt.get_cmap("viridis")

In [ ]:
global_min = adata_filtered.obs["chemokines_score"].min()
global_max = adata_filtered.obs["chemokines_score"].max()

print(global_min)
print(global_max)

### SCLE1

In [ ]:
gdf = scle1["cell_boundaries"].copy()
gdf.index = "SCLE1_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle1_cell_type.obs.loc[idx, "chemokines_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("chemokines_score")

plt.tight_layout()
plt.savefig(
    "SCLE1_chemokines_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE2

In [ ]:
gdf = scle2["cell_boundaries"].copy()
gdf.index = "SCLE2_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle2_cell_type.obs.loc[idx, "chemokines_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("chemokines_score")

plt.tight_layout()
plt.savefig(
    "SCLE2_chemokines_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE3

In [ ]:
gdf = scle3["cell_boundaries"].copy()
gdf.index = "SCLE3_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle3_cell_type.obs.loc[idx, "chemokines_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("chemokines_score")

plt.tight_layout()
plt.savefig(
    "SCLE3_chemokines_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE4

In [ ]:
gdf = scle4["cell_boundaries"].copy()
gdf.index = "SCLE4_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle4_cell_type.obs.loc[idx, "chemokines_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("chemokines_score")

plt.tight_layout()
plt.savefig(
    "SCLE4_chemokines_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE5

In [ ]:
gdf = scle5["cell_boundaries"].copy()
gdf.index = "SCLE5_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle5_cell_type.obs.loc[idx, "chemokines_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("chemokines_score")

plt.tight_layout()
plt.savefig(
    "SCLE5_chemokines_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

## D

In [ ]:
score_col = "remodel_score"   # column in adata.obs
celltype_col = "Cell_Type2_Clean"      # column in adata.obs
condition_col = "Lesion1"     # column in adata.obs
colors = {"SCLE": "#ff7f0e",
          "irAE-SCLE": "#279e68"}
order = ["Keratinocytes",
         "T Cells",
         "Macrophages",
          "Fibroblasts",
          "Endothelial Cells",]

# Make sure the needed columns exist
df = scle.obs[[score_col, celltype_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, celltype_col, condition_col])

plt.figure(figsize=(13, 7))
ax=sns.violinplot(
    data=df,
    x=celltype_col,
    y=score_col,
    hue=condition_col,
    order=order,
    split=True,          # split violin
    inner=None,
    cut=0,
    linewidth=0.5,
    palette=colors
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=17)
plt.yticks(size=20)
plt.xlabel(None)
plt.ylabel("ECM Degradation Score", size=25)
plt.title(None)
plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=15, title_fontsize=20)
plt.tight_layout()
plt.savefig('Degradation_SCLE.pdf')
plt.show()

## E

In [ ]:
global_min = np.nanpercentile(adata_filtered.obs["remodel_score"], 1)
global_max = np.nanpercentile(adata_filtered.obs["remodel_score"], 99)
# global_max = adata_filtered.obs["hypoxia_score"].max()
# global_min = adata_filtered.obs["hypoxia_score"].max()

print(global_min)
print(global_max)

### SCLE1

In [ ]:
gdf = scle1["cell_boundaries"].copy()
gdf.index = "SCLE1_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle1_cell_type.obs.loc[idx, "remodel_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("remodel_score")

plt.tight_layout()
plt.savefig(
    "SCLE1_remodel_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE2

In [ ]:
gdf = scle2["cell_boundaries"].copy()
gdf.index = "SCLE2_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle2_cell_type.obs.loc[idx, "remodel_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("remodel_score")

plt.tight_layout()
plt.savefig(
    "SCLE2_remodel_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE3

In [ ]:
gdf = scle3["cell_boundaries"].copy()
gdf.index = "SCLE3_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle3_cell_type.obs.loc[idx, "remodel_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("remodel_score")

plt.tight_layout()
plt.savefig(
    "SCLE3_remodel_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE4

In [ ]:
gdf = scle4["cell_boundaries"].copy()
gdf.index = "SCLE4_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle4_cell_type.obs.loc[idx, "remodel_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("remodel_score")

plt.tight_layout()
plt.savefig(
    "SCLE4_remodel_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE5

In [ ]:
gdf = scle5["cell_boundaries"].copy()
gdf.index = "SCLE5_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle5_cell_type.obs.loc[idx, "remodel_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("remodel_score")

plt.tight_layout()
plt.savefig(
    "SCLE5_remodel_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

# Supplementary Figure 1

In [ ]:
mapping_cell_type2 = {
    "Activated vECs": "Endothelial Cells (vECs)",
    "Adipocytes": "Adipocytes",
    "Basal Cells": "Keratinocytes",
    "CXCL9/10/11+ IFN Basal Cells": "Keratinocytes",
    "CXCL9/10/11+ IFN Fibroblasts": "Fibroblasts",
    "CXCL9/10/11+ IFN vECs": "Endothelial Cells (vECs)",
    "ECM Remodeling Fibroblasts": "Fibroblasts",
    "Eccrine Ductal Cells": "Adnexa (EDCs)",
    "Eccrine Gland Cells": "Adnexa (EGCs)",
    "Granular Keratinocytes": "Keratinocytes",
    "Hair Follicle Epithelia": "Adnexa (HF Epithelia)",
    "Hair Follicle-associated Fibroblasts": "Fibroblasts",
    "IFN Basal Cells": "Keratinocytes",
    "IFN Fibroblasts": "Fibroblasts",
    "IFN Spinous Keratinocytes": "Keratinocytes",
    "IFN T Cells": "Lymphoid Cells (T Cells)",
    "IFN vECs": "Endothelial Cells (vECs)",
    "Injury-associated Keratinocytes": "Keratinocytes",
    "Langerhans": "Myeloid Cells (Langerhans)",
    "Macrophages": "Myeloid Cells (Macrophages)",
    "Mast Cells": "Mast Cells",
    "Melanocytes": "Melanocytes",
    "Neutrophils": "Myeloid Cells (Neutrophils)",
    "Papillary Fibroblasts": "Fibroblasts",
    "Pericytes": "Pericytes",
    "Perineural Fibroblasts": "Fibroblasts",
    "Perivascular Fibroblasts": "Fibroblasts",
    "Reticular Fibroblasts": "Fibroblasts",
    "Schwann Cells": "Schwann Cells",
    "Sebocytes": "Adnexa (Sebocytes)",
    "Spinous Keratinocytes": "Keratinocytes",
    "T Cells": "Lymphoid Cells (T Cells)",
    "cDC1s": "Myeloid Cells (cDC1s)",
    "cDC2s": "Myeloid Cells (cDC2s)",
    "lECs": "Endothelial Cells (lECs)",
    "mregDCs": "Myeloid Cells (mregDCs)",
    "pDCs": "Myeloid Cells (pDCs)",
    "vECs": "Endothelial Cells (vECs)",
    "vSMCs": "Endothelial Cells (vSMCs)"
}

adata_filtered.obs['Cell_Type2_Clean'] = adata_filtered.obs['Cell_Type2'].map(mapping_cell_type2)

all_genes=["ADIPOQ","PLIN1", # Adipocytes
           "FADS2","FASN", #Sebocytes
           "SFRP1","SLC12A2", # Adnexa (Eccine Gland Cells)
           "SEMA3C","SESN3",# Adnexa (Eccrine Ductal Cells)
           "SOX9", #Hair Follicle Epithelia
           "DSG1","DMKN", #Keratinocytes (Suprabasal)
           "COL17A1", # Keratinocytes (Basal)
           "PECAM1","CD34", #vECs
           "FLT4","PROX1", # lECs
           "MYLK", "SMTN", #vSMC
           "RGS5","NOTCH3",#Pericytes
           "PDGFRA","PDGFRB", # Fibroblasts
           "CD3E", #Lymphoid Cells
           "CTSG","GATA2", #Mast Cells
           "MITF","MLANA",#Melanocytes
           "MRC1","F13A1", #Myeloid Cells (Macrophages)
           "CSF3R","ITGAX", #Myeloid Cells (Neutrophils)
           "CLEC10A", #Myeloid Cells (cDC2s)
           "XCR1","CLEC9A", #Myeloid Cells (cDC1s)
           "CLEC4C","IRF8", #Myeloid Cells (pDCs)
           "LAMP3","IRF4", #Myeloid Cells (mregDCs)
           "CD207","FCGBP", #Myeloid Cells (Langerhans)
           "MPZ","SCN7A",#Schwann Cells
           ]
adata_filtered.obs["Cell_Type2_Clean"] = (
    adata_filtered.obs["Cell_Type2_Clean"]
    .astype("category")
    .cat.reorder_categories(["Adipocytes","Adnexa (Sebocytes)","Adnexa (EGCs)", "Adnexa (EDCs)","Adnexa (HF Epithelia)","Keratinocytes", "Endothelial Cells (vECs)",
                             "Endothelial Cells (lECs)","Endothelial Cells (vSMCs)","Pericytes", "Fibroblasts", "Lymphoid Cells (T Cells)",
                             "Mast Cells", "Melanocytes", "Myeloid Cells (Macrophages)", "Myeloid Cells (Neutrophils)",
                             "Myeloid Cells (cDC2s)", "Myeloid Cells (cDC1s)", "Myeloid Cells (pDCs)",
                             "Myeloid Cells (mregDCs)","Myeloid Cells (Langerhans)",
                             "Schwann Cells"
                             ], ordered=True)
)
sc.pl.dotplot(adata_filtered, var_names=all_genes, groupby="Cell_Type2_Clean", save="_All")

In [ ]:
cell_type_column = 'Cell_Type_Clean'
group_column = 'Sample'

colors = {"Adipocytes": '#1f77b4',
          "Adnexa":'#C9A24D',
          "Endothelial Cells":'#5F9FD6',
          "Fibroblasts":'#4B50C8',
          "Keratinocytes":'#2A788E',
          "Lymphoid Cells":'#C24AD6',
          "Mast Cells":'#b5bd61',
          "Melanocytes":'#c5b0d5',
          "Myeloid Cells":'#D55E00',
          "Pericytes":'#aec7e8',
          "Schwann Cells":'#ffbb78',}

# Step 1: Aggregate data
cell_type_counts = (
    adata_filtered.obs.groupby([group_column, cell_type_column])
    .size()
    .unstack(fill_value=0)
)
cell_type_proportions = cell_type_counts.div(cell_type_counts.sum(axis=1), axis=0)

desired_order = ["NS1","NS2","NS3","NS4",'irAE-SCLE1','irAE-SCLE2','SCLE1','SCLE2','SCLE3']

# Reorder the rows in the DataFrame based on the desired order
cell_type_proportions_ordered = cell_type_proportions.reindex(desired_order)
cell_type_proportions_ordered = cell_type_proportions_ordered[
    cell_type_proportions_ordered.columns[::-1]
]
# Step 2: Stacked Bar Plot
ax = cell_type_proportions_ordered.plot(
    kind='bar',
    stacked=True,
    figsize=(20, 10),
    color=colors
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          title='Cell Type',
          bbox_to_anchor=(1, 1),
          fontsize=20,
          title_fontsize=25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=0, size=20)
plt.yticks(size=20)
plt.xlabel('')
plt.ylabel('Proportion', size=30)
# plt.title('Cell Type Composition')
plt.tight_layout()
plt.grid(False)
plt.savefig('Cell_Type_Comp_by_Sample.pdf')
plt.show()

# Supplementary Figure 2

In [ ]:
irae = adata_filtered[adata_filtered.obs['Lesion1']=="irAE-SCLE"].copy()

sc.pp.pca(irae, random_state=42)
sc.pp.neighbors(irae, random_state=42)
sc.tl.umap(irae, random_state=42)

In [ ]:
sc.pl.umap(irae, color=['Sample',"Cell_Type","Cell_Type1"], size=10, ncols=2)

In [ ]:
sc.set_figure_params(figsize=(10,10), dpi=300, dpi_save=300, transparent=True)

In [ ]:
sc.pl.umap(irae, color="Sample", size=30, title="", frameon=False,save="_irAE_Sample")
sc.pl.umap(irae, color="Cell_Type_Clean", size=30, title="", frameon=False,save="_irAE_Cell_Type")

# Supplementary Figure 3

In [ ]:
sc.pl.umap(tcell, color=['CD3E',"CD3G"], size=30)

In [ ]:
sc.pl.umap(tcell, color=["Cell_Type_Clean1","CD3E","CD8A","CD8B","CD4","TRDC","FOXP3","CTLA4"], 
           size=60, title="", ncols=2,vmax="p99", frameon=False, save="_Tcell_Genes"
           )

# Supplementary Figure 4

In [ ]:
score_col = "ifn_sig_score"   # column in adata.obs
condition_col = "Sample"     # column in adata.obs
order = ["NS1","NS2","NS3","NS4",'irAE-SCLE1','irAE-SCLE2','SCLE1','SCLE2','SCLE3']
# Make sure the needed columns exist
df = adata_filtered.obs[[score_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, condition_col])

plt.figure(figsize=(20, 10))
ax=sns.violinplot(
    data=df,
    x=condition_col,
    y=score_col,
    hue=condition_col,
    order=order,         # split violin
    inner=None,
    cut=0,
    linewidth=1,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel(None)
plt.ylabel("IFN Signaling Score", size=35)
plt.title(None)
# plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=20, title_fontsize=25)
plt.tight_layout()
plt.savefig('IFN_Sig_by_Sample.pdf')
plt.show()

In [ ]:
score_col = "isg_score"   # column in adata.obs
condition_col = "Sample"     # column in adata.obs
order = ["NS1","NS2","NS3","NS4",'irAE-SCLE1','irAE-SCLE2','SCLE1','SCLE2','SCLE3']
# Make sure the needed columns exist
df = adata_filtered.obs[[score_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, condition_col])

plt.figure(figsize=(20, 10))
ax=sns.violinplot(
    data=df,
    x=condition_col,
    y=score_col,
    hue=condition_col,
    order=order,         # split violin
    inner=None,
    cut=0,
    linewidth=1,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel(None)
plt.ylabel("ISG Score", size=35)
plt.title(None)
# plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=20, title_fontsize=25)
plt.tight_layout()
plt.savefig('ISG_by_Sample.pdf')
plt.show()

# Supplementary Figure 5

In [ ]:
for i in scle_superficial.obs['Sample'].unique():
    print(i,"\n", scle_superficial[scle_superficial.obs["Sample"]==i].obs['Cell_Type1'].value_counts())

In [ ]:
print(469/358)
print(1124/943)
print(3362/1214)
print(2148/440)
print(4911/1219)

In [ ]:
samples = ["irAE-SCLE1", "irAE-SCLE2", "SCLE1", "SCLE2", "SCLE3"]
ratios = [
    1.3100558659217878,
    1.1919406150583245,
    2.769357495881384,
    4.881818181818182,
    4.028712059064807
]

# Use the SAME palette as your violin plot
order = ["NS1","NS2","NS3","NS4",
         "irAE-SCLE1","irAE-SCLE2",
         "SCLE1","SCLE2","SCLE3"]

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    samples,
    ratios,
    color=[palette[7], palette[8], palette[4], palette[5], palette[6]],
    edgecolor="black",
    linewidth=1
)

# Turn off grid
ax.grid(False)

# Hide top and right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylabel("Lymphoid : Myeloid Ratio")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig('Lymphoid_Myeloid_Ratio.pdf')
plt.show()

# Supplementary Figure 6

In [ ]:
score_col = "hypoxia_score"   # column in adata.obs
celltype_col = "Cell_Type2_Clean"      # column in adata.obs
condition_col = "Lesion1"     # column in adata.obs
colors = {"SCLE": "#ff7f0e",
          "irAE-SCLE": "#279e68"}
order = ["Keratinocytes",
         "T Cells",
         "Macrophages",
          "Fibroblasts",
          "Endothelial Cells",]

# Make sure the needed columns exist
df = scle.obs[[score_col, celltype_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, celltype_col, condition_col])

plt.figure(figsize=(18, 7))
ax=sns.violinplot(
    data=df,
    x=celltype_col,
    y=score_col,
    hue=condition_col,
    order=order,
    split=True,          # split violin
    inner=None,
    cut=0,
    linewidth=0.5,
    palette=colors
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel(None)
plt.ylabel("Hypoxia Score", size=30)
plt.title(None)
plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=25, title_fontsize=30)
plt.tight_layout()
plt.savefig('Hypoxia_SCLE.pdf')
plt.show()

In [ ]:
score_col = "hypoxia_score"   # column in adata.obs
condition_col = "Sample"     # column in adata.obs
order = ["NS1","NS2","NS3","NS4",'irAE-SCLE1','irAE-SCLE2','SCLE1','SCLE2','SCLE3']
# Make sure the needed columns exist
df = adata_filtered.obs[[score_col, condition_col]].copy()

# Optional: remove missing values
df = df.dropna(subset=[score_col, condition_col])

plt.figure(figsize=(20, 10))
ax=sns.violinplot(
    data=df,
    x=condition_col,
    y=score_col,
    hue=condition_col,
    order=order,         # split violin
    inner=None,
    cut=0,
    linewidth=1,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', linewidth=0.5, alpha=0.3)
plt.xticks(rotation=0, size=25)
plt.yticks(size=25)
plt.xlabel(None)
plt.ylabel("Hypoxia Score", size=35)
plt.title(None)
# plt.legend(title="Lesion", bbox_to_anchor=(1, 1), fontsize=20, title_fontsize=25)
plt.tight_layout()
plt.savefig('Hypoxia_by_Sample.pdf')
plt.show()

In [ ]:
global_min = np.nanpercentile(adata_filtered.obs["hypoxia_score"], 1)
global_max = np.nanpercentile(adata_filtered.obs["hypoxia_score"], 99.9)
# global_max = adata_filtered.obs["hypoxia_score"].max()
# global_min = adata_filtered.obs["hypoxia_score"].max()

print(global_min)
print(global_max)

### SCLE1

In [ ]:
gdf = scle1["cell_boundaries"].copy()
gdf.index = "irAE-SCLE1_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle1_cell_type.obs.loc[idx, "hypoxia_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("hypoxia_score")

plt.tight_layout()
plt.savefig(
    "SCLE1_hypoxia_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE2

In [ ]:
gdf = scle2["cell_boundaries"].copy()
gdf.index = "irAE-SCLE2_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle2_cell_type.obs.loc[idx, "hypoxia_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("hypoxia_score")

plt.tight_layout()
plt.savefig(
    "SCLE2_hypoxia_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE3

In [ ]:
gdf = scle3["cell_boundaries"].copy()
gdf.index = "SCLE1_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle3_cell_type.obs.loc[idx, "hypoxia_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("hypoxia_score")

plt.tight_layout()
plt.savefig(
    "SCLE3_hypoxia_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE4

In [ ]:
gdf = scle4["cell_boundaries"].copy()
gdf.index = "SCLE2_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle4_cell_type.obs.loc[idx, "hypoxia_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("hypoxia_score")

plt.tight_layout()
plt.savefig(
    "SCLE4_hypoxia_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()

### SCLE5

In [ ]:
gdf = scle5["cell_boundaries"].copy()
gdf.index = "SCLE3_" + gdf.index

patches = []
scores = []

# collect patches and scores
for idx, row in gdf.iterrows():
    geom = row.geometry
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]

    # pull score once per cell
    try:
        score = scle5_cell_type.obs.loc[idx, "hypoxia_score"]
    except Exception:
        score = np.nan

    for poly in polys:
        poly_scaled = scale(poly, xfact=2, yfact=2, origin="centroid")
        exterior_coords = np.asarray(poly_scaled.exterior.coords)
        patches.append(MplPolygon(exterior_coords, closed=True))
        scores.append(score)

# convert scores to colors
scores_arr = np.array(scores, dtype=float)
norm = Normalize(vmin=global_min, vmax=global_max)
facecolors = cmap(norm(scores_arr))
facecolors[np.isnan(scores_arr)] = [1, 1, 1, 1] 

# plotting
fig, ax = plt.subplots(figsize=(6, 10))
pc = PatchCollection(patches, match_original=False)
pc.set_facecolor(facecolors)
pc.set_edgecolor('none')
pc.set_alpha(0.9)
ax.add_collection(pc)

ax.autoscale_view()
ax.set_aspect('equal', adjustable='datalim')
ax.invert_yaxis()
ax.axis('off')

# colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("hypoxia_score")

plt.tight_layout()
plt.savefig(
    "SCLE5_hypoxia_score.pdf",
    dpi=300,
    transparent=True
)
plt.show()